In [70]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

def add_user_messages(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_messages(messages, message):
    messages.append({"role": "assistant", "content": message})
    
def chat(messages, tools=[], stop_sequences=[], temperature=1.0, system=None):
    client = anthropic.Anthropic()
    parameters = {
        "model": "claude-haiku-4-5",
        "max_tokens": 512,
        "tools": tools,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        parameters["system"] = system

    response = client.messages.create(**parameters)
    # print(response)
    return response
    # return response.content[0].text

In [71]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(format="%Y-%m-%d %H:%M:%S"):
    if not format:
        raise ValueError("Format string cannot be empty")
    return datetime.now().strftime(format)

get_current_datetime_schema = {
        "name": "get_current_datetime",
        "description": "Retrieves the current date and time in the specified format. This tool is useful when you need to know the present date, time, or both. You can customize the output format using Python strftime syntax (e.g., '%Y-%m-%d' for just the date, '%H:%M:%S' for just the time). If no format is provided, it defaults to 'YYYY-MM-DD HH:MM:SS' format. The tool will raise an error if an empty format string is provided.",
        "input_schema": {
            "type": "object",
            "properties": {
            "format": {
                "type": "string",
                "description": "Python strftime format string to control the output format. Common examples: '%Y-%m-%d' (2024-05-27), '%H:%M:%S' (14:30:45), '%Y-%m-%d %H:%M:%S' (2024-05-27 14:30:45), '%A, %B %d, %Y' (Monday, May 27, 2024). Defaults to '%Y-%m-%d %H:%M:%S' if not provided. Cannot be an empty string.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
            },
            "required": []
        },
        "input_examples": [
            {
            "format": "%Y-%m-%d"
            },
            {
            "format": "%H:%M:%S"
            },
            {
            "format": "%A, %B %d, %Y at %I:%M %p"
            },
            {}
        ]
    }

In [72]:
messages = []
add_user_messages(messages, "What is the current date and time in ISO format?")
response = chat(messages, tools=[get_current_datetime_schema])
add_assistant_messages(messages, response.content)
now = get_current_datetime(**response.content[0].input)
add_user_messages(messages, 
    [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": now,
            "is_error": False
        }
    ]
)
response = chat(messages)
response

Message(id='msg_01DkppDWzjFk7i3CZYhu5UQU', container=None, content=[TextBlock(citations=None, text='The current date and time in ISO format is: **2026-05-28T13:05:36**', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=110, output_tokens=27, server_tool_use=None, service_tier='standard'))